In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path("..").resolve()))

import pandas as pd
from src.classify import train_text_classifier, evaluate_classifier, save_model

train_df = pd.read_csv("../data/processed/train_clean.csv")
val_df = pd.read_csv("../data/processed/validation_clean.csv")

In [2]:
class_action_model = train_text_classifier(
    train_df,
    label_col="class_action_sought"
)

class_action_results = evaluate_classifier(
    class_action_model,
    val_df,
    label_col="class_action_sought"
)

print(class_action_results["accuracy"])
print(class_action_results["macro_f1"])
print(class_action_results["classification_report"])

0.9273127753303965
0.6141466436092702
              precision    recall  f1-score   support

          No       0.95      0.94      0.94       298
     Unknown       0.00      0.00      0.00         1
         Yes       0.89      0.91      0.90       155

    accuracy                           0.93       454
   macro avg       0.61      0.62      0.61       454
weighted avg       0.93      0.93      0.93       454



D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_d

In [3]:
save_model(
    class_action_model,
    "../models/class_action/class_action_tfidf_logreg.pkl"
)

In [ ]:
# take away unknown

In [4]:
train_binary = train_df[train_df["class_action_sought"].isin(["Yes", "No"])]
val_binary = val_df[val_df["class_action_sought"].isin(["Yes", "No"])]

class_action_model = train_text_classifier(
    train_binary,
    label_col="class_action_sought"
)

class_action_results = evaluate_classifier(
    class_action_model,
    val_binary,
    label_col="class_action_sought"
)

print(class_action_results["accuracy"])
print(class_action_results["macro_f1"])
print(class_action_results["classification_report"])

0.9293598233995585
0.9224746502374535
              precision    recall  f1-score   support

          No       0.96      0.93      0.95       298
         Yes       0.88      0.92      0.90       155

    accuracy                           0.93       453
   macro avg       0.92      0.93      0.92       453
weighted avg       0.93      0.93      0.93       453



In [5]:
save_model(
    class_action_model,
    "../models/class_action/class_action_tfidf_logreg.pkl"
)

In [6]:
train_df["case_type"].value_counts().head(10)

case_type
Equal Employment                         1099
Immigration and/or the Border             367
Prison Conditions                         313
Jail Conditions                           223
Public Benefits / Government Services     204
Policing                                  143
Speech and Religious Freedom              134
Criminal Justice (Other)                  120
National Security                          94
Fair Housing/Lending/Insurance             84
Name: count, dtype: int64

In [ ]:
# 5-way classification

In [7]:
top_case_types = train_df["case_type"].value_counts().head(5).index.tolist()

top_case_types

['Equal Employment',
 'Immigration and/or the Border ',
 'Prison Conditions',
 'Jail Conditions',
 'Public Benefits / Government Services']

In [8]:
train_case = train_df[
    train_df["case_type"].isin(top_case_types)
]

val_case = val_df[
    val_df["case_type"].isin(top_case_types)
]

In [9]:
case_type_model = train_text_classifier(
    train_case,
    label_col="case_type"
)

case_type_results = evaluate_classifier(
    case_type_model,
    val_case,
    label_col="case_type"
)

print(case_type_results["accuracy"])
print(case_type_results["macro_f1"])
print(case_type_results["classification_report"])

0.9464882943143813
0.9134463472127802
                                       precision    recall  f1-score   support

                     Equal Employment       0.99      0.99      0.99       155
       Immigration and/or the Border        0.96      0.92      0.94        60
                      Jail Conditions       0.87      0.91      0.89        22
                    Prison Conditions       0.93      0.90      0.92        30
Public Benefits / Government Services       0.80      0.88      0.84        32

                             accuracy                           0.95       299
                            macro avg       0.91      0.92      0.91       299
                         weighted avg       0.95      0.95      0.95       299



In [ ]:
# short/tiny summary baseline

In [10]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [12]:
import pandas as pd

train_clean = pd.read_csv("../data/processed/train_clean.csv")

In [13]:
train_clean.head()

,id,case_name,court,date_filed,case_type,class_action_sought,summary/long,summary/short,summary/tiny,full_text,text_length
0,EE-AL-0045,"EEOC v. House of Philadelphia Center, Inc.",NaN,NaN,Equal Employment,No,"On September 15, 2005, the Equal Employment Op...",Equal Employment Opportunity Commission brough...,NaN,Case 1:05-cv-00530-D Document 1-1 Filed 09/19/...,32174
1,PB-NJ-0003,Disability Rights New Jersey v. Velez,NaN,NaN,Public Benefits / Government Services,No,NOTE: This is one of three identically named ...,The case was brought by a non-profit organizat...,NaN,Case 3:05-cv-01784-SRC-JJH Document 2 Filed 05...,152724
2,EE-FL-0136,United States v. Palm Beach County,NaN,NaN,Equal Employment,No,"On August 9, 2007, the United States Departmen...",NaN,NaN,Case 9:07-cv-80713-KAM Document 1 Entered on F...,41712
3,EE-CA-0305,Wynne v. McCormick & Schmick's Seafood Restara...,NaN,NaN,Equal Employment,Yes,"On May 11, 2006, African-American employees of...",This case was brought by African American empl...,NaN,2006 WL 1787244\n2006 WL 1787244 (N.D.Cal.) (T...,367032
4,NH-NJ-0002,"U.S. v. Mercer County, New Jersey",NaN,NaN,Nursing Home Conditions,No,Pursuant to the Civil Rights of Institutionali...,Pursuant to the Civil Rights of Institutionali...,NaN,IN THE UNITED STATES DISTRICT COURT FOR THE DI...,105806


In [14]:
sample = train_clean.iloc[0]

long_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="long"
)

short_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="short"
)

tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print("LONG\n", long_summary[:1000])
print("\nSHORT\n", short_summary)
print("\nTINY\n", tiny_summary)

LONG
 f scx, femalc, by discharging Ms. Griffin due to her pregnancy, in

violation of Title VII.

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of House of Philadelphia.

The Parties do not object to the jurisdiction of the Court over this action and waive their rights to a hearing and the entry of findings of fact and conclusions of law. Venue is appropriate in the Southern District of Alabama (Southern Division). The parties agree that this Consent Decree is fair, reasonable, and does not violate the law or public policy. The rights of Ms. Griffin, House of Philadelphia, and the Commission are protected adequately by this Decree.
In the interest of resolving this matter~ avoiding the expense of further litigation, and as a result of having engaged in comprehensive settlement negotiations, the Commission and House of Philadelphia

In [15]:
import importlib
import src.summarize
importlib.reload(src.summarize)

from src.summarize import summarize_document

In [16]:
sample = train_clean.iloc[0]

long_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="long"
)

short_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="short"
)

tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print("LONG\n", long_summary[:1000])
print("\nSHORT\n", short_summary)
print("\nTINY\n", tiny_summary)

LONG
 f scx, femalc, by discharging Ms. Griffin due to her pregnancy, in

violation of Title VII.

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of House of Philadelphia.

The Parties do not object to the jurisdiction of the Court over this action and waive their rights to a hearing and the entry of findings of fact and conclusions of law. Venue is appropriate in the Southern District of Alabama (Southern Division). The parties agree that this Consent Decree is fair, reasonable, and does not violate the law or public policy. The rights of Ms. Griffin, House of Philadelphia, and the Commission are protected adequately by this Decree.
In the interest of resolving this matter~ avoiding the expense of further litigation, and as a result of having engaged in comprehensive settlement negotiations, the Commission and House of Philadelphia

In [17]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [18]:
tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print(tiny_summary)

Griffin due to her pregnancy, in

violation of Title VII.

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of House of Philadelphia.

The Parties do not object to the jurisdiction of the Court over this action and waive their rights to a hearing and the entry of findings of fact and conclusions of law.


In [19]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [20]:
tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print(tiny_summary)

Griffin due to her pregnancy, in

violation of Title VII..


In [23]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [24]:
tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print(tiny_summary)

Defendant and its officers, agents, employees, successors, and assigns both at the time that this Decree becomes effective and for the duration of this Decree agree to comply with Federal law and acknowledge that it is unlawful to: (a) discriminate against any employee on the basis of pregnancy or sex, (b) harass any employee based on pregnancy or sex; (c) retaliate against any employee because he or she: (i) opposes or opposed discriminatory practices nlade unlawful by Title VII; (ii) files or filed a charge of discrimination or assists, assisted,
2

participates, or participated in the filing of a charge of discrimination; or (iii) assists, assisted, participates or participated in an investigation or proceeding brought under the federal laws prohibiting discri


In [25]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [26]:
tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print(tiny_summary)

In the interest of resolving this matter~ avoiding the expense of further litigation, and as a result of having engaged in comprehensive settlement negotiations, the Commission and House of Philadelphia have agreed that this action should be finally resolved by entry of this Consent Decree.


In [27]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [28]:
tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print(tiny_summary)

Griffin due to her pregnancy, in

violation of Title VII.


In [29]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [30]:
tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print(tiny_summary)

Griffin due to her pregnancy, in

violation of Title VII.


In [31]:
import importlib
import src.summarize

importlib.reload(src.summarize)

from src.summarize import summarize_document

In [32]:
tiny_summary = summarize_document(
    sample["full_text"],
    reference_summary=sample["summary/long"],
    summary_type="tiny"
)

print(tiny_summary)

House of Philadelphia denies all allegations of unlawful or wrongful conduct raised in the

complaint, and nothing stated in this Decree constitutes an admission of liability or wrongdoing

on the part of House of Philadelphia.
